# FlameTrack — Analysis Pipeline Walkthrough

This notebook replicates **exactly** what the FlameTrack application does, step by step:

1. Load a dewarped IR frame from an HDF5 result file
2. Background subtraction (frame – previous frame)
3. Otsu masking — restrict the per-row search window
4. Edge detection on a single row — all methods compared
5. Full-frame edge result overlaid on the thermal image
6. Edge position over time (flame spread curve)

**Where the code lives:**  
- Dewarping → `src/flametrack/processing/dewarping.py`  
- Edge detection → `src/flametrack/analysis/flamespread.py`, function `calculate_edge_data`

## 0 · Configuration

Set `HDF5_PATH` to your result file, or leave `USE_SYNTHETIC = True` to run on generated demo data.

In [ ]:
# ── User settings ──────────────────────────────────────────────────────────────
USE_SYNTHETIC = True  # set False to load a real HDF5 file
HDF5_PATH = "../path/to/experiment_results_RCE.h5"
DATASET_KEY = "dewarped_data"  # 'dewarped_data', 'dewarped_data_left', …

FRAME_IDX = 10  # which frame to visualise
ROW_IDX = 80  # which row (y) to show the 1-D profile for
THRESHOLD_IR = 280.0  # edge threshold (K for IR data)
FLAME_DIR = "right_to_left"  # 'right_to_left' or 'left_to_right'

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join("..", "src"))

import cv2
import h5py
import matplotlib.pyplot as plt
import numpy as np

from flametrack.analysis.flamespread import (
    EDGE_METHOD_CATALOG,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 10
plt.rcParams["axes.labelsize"] = 9

## 1 · Load data

The dewarped data array has shape **(H, W, T)** — height × width × frames.
This is written by `dewarp_room_corner_remap` / `dewarp_lateral_flame_spread` in `dewarping.py`.

In [ ]:
def make_synthetic_data(H=160, W=280, T=30, seed=0):
    """Generate a fake IR sequence of a flame spreading right→left."""
    rng = np.random.default_rng(seed)
    data = np.full((H, W, T), 290.0, dtype=np.float32)  # ambient ~290 K
    for t in range(T):
        edge_x = W - 1 - int((t / T) * (W * 0.65))  # edge moves left
        flame = data[:, :, t]
        flame[:, edge_x:] = 580.0 + rng.normal(0, 8, (H, W - edge_x))
        # gradient at edge
        for dx in range(1, 12):
            if edge_x - dx >= 0:
                flame[:, edge_x - dx] = 290 + (580 - 290) * np.exp(-dx / 3)
        # add a few artefact hot spots in the background
        for _ in range(3):
            rx = rng.integers(0, max(edge_x - 20, 1))
            ry = rng.integers(0, H)
            flame[ry, rx] = 320.0
    return data


if USE_SYNTHETIC:
    data = make_synthetic_data()
    print(f"Synthetic data  shape={data.shape}  dtype={data.dtype}")
    print(f"  min={data.min():.1f} K   max={data.max():.1f} K")
else:
    with h5py.File(HDF5_PATH, "r") as f:
        data = f[DATASET_KEY]["data"][:]  # load fully into RAM
    print(f"Loaded {HDF5_PATH}")
    print(f"  shape={data.shape}  dtype={data.dtype}")
    print(f"  min={data.min():.1f}   max={data.max():.1f}")

H, W, T = data.shape
FRAME_IDX = min(FRAME_IDX, T - 1)
ROW_IDX = min(ROW_IDX, H - 1)

## 2 · Raw dewarped frame

This is what the image canvas shows in the app **after dewarping** — perspective distortion is already corrected, and the specimen plate fills the entire image.

In [ ]:
frame_raw = data[:, :, FRAME_IDX].astype(np.float32)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(frame_raw, cmap="inferno", aspect="auto")
ax.axhline(ROW_IDX, color="cyan", linewidth=1.2, label=f"row {ROW_IDX} (used later)")
ax.set_title(f"Dewarped IR frame {FRAME_IDX}  —  shape {frame_raw.shape}")
ax.set_xlabel("x (pixels)")
ax.set_ylabel("y (row)")
ax.legend(loc="upper left", fontsize=8)
fig.colorbar(im, ax=ax, label="Temperature (K)")
plt.tight_layout()
plt.show()

## 3 · Background subtraction

**Code in `calculate_edge_data`:**
```python
filtered_frame    = custom_filter(frame.copy())
frame             = filtered_frame - custom_filter(background_frame)
```
The previous frame is subtracted to suppress the static background and highlight *changes* (i.e. the advancing flame front).

> **Note:** In the current app code the background-subtracted `frame` variable is computed here but is **not used** in the per-row edge search — both Otsu masking and the edge function receive `filtered_frame` (the non-subtracted version). The subtraction is a remnant of an earlier approach and is a candidate for future cleanup.

In [ ]:
background_raw = data[:, :, max(FRAME_IDX - 1, 0)].astype(np.float32)

custom_filter = lambda x: x  # default in the app: identity
filtered_frame = custom_filter(frame_raw.copy())
diff_frame = filtered_frame - custom_filter(background_raw)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

for ax, img, title in zip(
    axes,
    [background_raw, filtered_frame, diff_frame],
    [
        f"Previous frame (frame {max(FRAME_IDX - 1, 0)})",
        f"Current frame (frame {FRAME_IDX})",
        "Difference  (current − previous)",
    ],
):
    im = ax.imshow(img, cmap="inferno", aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    fig.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("Background subtraction  (step 1 in calculate_edge_data)", y=1.02)
plt.tight_layout()
plt.show()

## 4 · Otsu masking

**Purpose:** Instead of scanning every pixel in a row, Otsu's method automatically finds a global threshold that separates *background* from *hot region*. A 10-iteration dilation then expands the mask slightly so the edge at the boundary is not excluded.

**Code in `calculate_edge_data`:**
```python
# 1. Normalise filtered_frame to [0, 255]
normed  = (filtered_frame - min) / (max - min)
mask_u8 = (normed * 255).astype(np.uint8)

# 2. Otsu threshold
_, thresh = cv2.threshold(mask_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 3. Dilate to capture the edge zone
kernel = np.ones((3, 3), np.uint8)
thresh = cv2.dilate(thresh, kernel, iterations=10)
```
Per row the first and last non-zero column of `thresh` become `start` and `end` — the search window passed to the edge function.

In [ ]:
minv = float(filtered_frame.min())
maxv = float(filtered_frame.max())
normed = (filtered_frame - minv) / (maxv - minv)
mask_u8 = (normed * 255.0).astype(np.uint8)

otsu_val, thresh_raw = cv2.threshold(
    mask_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

kernel = np.ones((3, 3), dtype=np.uint8)
thresh_dilated = cv2.dilate(thresh_raw, kernel, iterations=10)

print(f"Otsu threshold (on 0–255 scale): {otsu_val}")
print(f"  ≈ {otsu_val / 255 * (maxv - minv) + minv:.1f} K in original units")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))

axes[0].imshow(filtered_frame, cmap="inferno", aspect="auto")
axes[0].set_title("filtered_frame (input to Otsu)")

axes[1].imshow(mask_u8, cmap="gray", aspect="auto")
axes[1].set_title("Normalised → uint8 [0–255]")

axes[2].imshow(thresh_raw, cmap="gray", aspect="auto")
axes[2].set_title(f"Otsu binary mask  (threshold={otsu_val})")

axes[3].imshow(thresh_dilated, cmap="gray", aspect="auto")
axes[3].set_title("After dilation (10 × 3×3 kernel)")

for ax in axes:
    ax.axhline(ROW_IDX, color="cyan", linewidth=1)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.suptitle("Otsu masking pipeline", y=1.02)
plt.tight_layout()
plt.show()

### 4a · Search window for the selected row

For each row the app reads `start` and `end` from the dilated mask and passes only `filtered_frame[row, start:end]` to the edge function.

In [ ]:
row_mask = thresh_dilated[ROW_IDX, :]
hot_idx = np.where(row_mask > 0)[0]

if hot_idx.size:
    start = int(hot_idx[0])
    end = int(hot_idx[-1]) + 10  # +10: same as in calculate_edge_data
else:
    start, end = 0, W

print(
    f"Row {ROW_IDX}: Otsu search window  start={start}  end={end}  (width={end - start} px)"
)

row_signal = filtered_frame[ROW_IDX, :]

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(row_signal, color="tomato", label="Temperature (K)")
ax.axvspan(0, start, alpha=0.12, color="gray", label="masked out (left)")
ax.axvspan(end, W, alpha=0.12, color="gray", label="masked out (right)")
ax.axvspan(start, end, alpha=0.12, color="gold", label="Otsu search window")
ax.axhline(
    THRESHOLD_IR,
    color="steelblue",
    linestyle="--",
    label=f"threshold = {THRESHOLD_IR} K",
)
ax.set_xlabel("x (pixel)")
ax.set_ylabel("Temperature (K)")
ax.set_title(f"Row {ROW_IDX} — 1-D intensity profile with Otsu window")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5 · Edge detection — all methods on one row

FlameTrack ships six edge-finding methods in `EDGE_METHOD_CATALOG`. All of them receive the **windowed** signal `y = filtered_frame[row, start:end]` and return an index *within that window* — the app then adds `start` to get the absolute pixel position.

| Method | Direction | Otsu window? | Strategy |
|--------|-----------|--------------|----------|
| `leftmost_threshold` | R→L | yes | First pixel ≥ threshold |
| `rightmost_threshold` | L→R | yes | Last pixel ≥ threshold |
| `left_edge_rightmost_cluster` | R→L | yes | Left edge of rightmost hot cluster |
| `right_edge_leftmost_cluster` | L→R | yes | Right edge of leftmost hot cluster |
| `leftmost_threshold_no_otsu` | R→L | no | First pixel ≥ threshold, full row |
| `rightmost_threshold_no_otsu` | L→R | no | Last pixel ≥ threshold, full row |

In [ ]:
y_windowed = filtered_frame[ROW_IDX, start:end]
y_full = filtered_frame[ROW_IDX, :]

colors = ["#e41a1c", "#377eb8", "#ff7f00", "#4daf4a", "#984ea3", "#a65628"]
markers = {}

fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharey=True)

for ax, (key, spec), color in zip(axes.flat, EDGE_METHOD_CATALOG.items(), colors):
    edge_fn = spec.make_fn(THRESHOLD_IR)

    if spec.use_otsu_masking:
        y_in = y_windowed
        offset = start
        ax.axvspan(0, start, alpha=0.1, color="gray")
        ax.axvspan(end, W, alpha=0.1, color="gray")
        ax.axvspan(start, end, alpha=0.12, color="gold")
    else:
        y_in = y_full
        offset = 0

    edge_local = edge_fn(y_in)
    edge_abs = edge_local + offset if edge_local > 0 else offset
    markers[key] = edge_abs

    ax.plot(y_full, color="tomato", linewidth=1.2, label="row signal")
    ax.axvline(edge_abs, color=color, linewidth=2, label=f"edge = {edge_abs} px")
    ax.axhline(
        THRESHOLD_IR,
        color="steelblue",
        linestyle="--",
        linewidth=0.8,
        label=f"threshold={THRESHOLD_IR}",
    )
    ax.set_title(spec.display_name, fontsize=8)
    ax.set_xlabel("x (px)", fontsize=7)
    ax.set_ylabel("K", fontsize=7)
    ax.legend(fontsize=7)

plt.suptitle(f"All 6 edge methods  —  row {ROW_IDX}, frame {FRAME_IDX}", y=1.02)
plt.tight_layout()
plt.show()

print("\nEdge positions (absolute pixel x):")
for k, v in markers.items():
    print(f"  {k:<35} {v}")

## 6 · Full-frame edge — all methods overlaid

Running the selected method on **every row** produces a column of edge positions that traces the flame contour across the plate.

In [ ]:
def run_one_frame(frame_idx, method_key="leftmost_threshold", threshold=THRESHOLD_IR):
    """Replicate the inner loop of calculate_edge_data for a single frame."""
    spec = EDGE_METHOD_CATALOG[method_key]
    fn = spec.make_fn(threshold)
    f_raw = data[:, :, frame_idx].astype(np.float32)
    filt = f_raw.copy()  # custom_filter = identity

    thresh_map = None
    if spec.use_otsu_masking:
        mn, mx = filt.min(), filt.max()
        u8 = ((filt - mn) / (mx - mn) * 255).astype(np.uint8)
        _, tm = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        thresh_map = cv2.dilate(tm, np.ones((3, 3), np.uint8), iterations=10)

    edges = []
    for i in range(filt.shape[0]):
        s, e = 0, -1
        if thresh_map is not None and frame_idx < 150:
            idx = np.where(thresh_map[i, :] > 0)[0]
            if idx.size:
                s, e = int(idx[0]), int(idx[-1]) + 10
        y = filt[i, s:e]
        p = fn(y)
        edges.append(p + s if p > 0 else s)
    return np.array(edges), f_raw


method_to_show = (
    "leftmost_threshold" if FLAME_DIR == "right_to_left" else "rightmost_threshold"
)
edges, frame_display = run_one_frame(FRAME_IDX, method_to_show)

fig, ax = plt.subplots(figsize=(9, 4))
ax.imshow(frame_display, cmap="inferno", aspect="auto")
ax.plot(edges, np.arange(H), color="cyan", linewidth=1.5, label="detected edge")
ax.axhline(
    ROW_IDX,
    color="yellow",
    linewidth=0.8,
    linestyle="--",
    label=f"row {ROW_IDX} (detail shown above)",
)
ax.set_title(
    f"Frame {FRAME_IDX}  —  method: {EDGE_METHOD_CATALOG[method_to_show].display_name}"
)
ax.set_xlabel("x (px)")
ax.set_ylabel("y (row)")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

## 7 · Edge over time — flame spread curve

Running the same loop over all T frames produces `edge_results` of shape **(T, H)**. The x-position at a fixed row traces how the flame front moves over time — this is the *flame spread curve* shown in the Analysis Plot of the app.

In [ ]:
print(f"Running edge detection on all {T} frames …")
all_edges = np.array([run_one_frame(t, method_to_show)[0] for t in range(T)])
# shape: (T, H)
print(f"edge_results shape: {all_edges.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# left: edge position matrix (heatmap)
im = axes[0].imshow(
    all_edges.T, aspect="auto", cmap="viridis", origin="upper", extent=[0, T, H, 0]
)
axes[0].axhline(ROW_IDX, color="white", linewidth=1, linestyle="--")
axes[0].set_xlabel("Frame")
axes[0].set_ylabel("Row (y)")
axes[0].set_title("Edge x-position over time  (all rows)")
fig.colorbar(im, ax=axes[0], label="Edge x (px)")

# right: flame spread at selected row
axes[1].plot(all_edges[:, ROW_IDX], color="tomato", linewidth=1.5)
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Edge x-position (px)")
axes[1].set_title(f"Flame spread at row {ROW_IDX}")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Flame spread results  —  equivalent to the Analysis Plot in the app")
plt.tight_layout()
plt.show()

## 8 · Summary

The complete pipeline in one picture:

```
HDF5 dewarped_data  (H × W × T)
       │
       ├─ for each frame t:
       │       filtered_frame = frame          ← custom_filter (default: identity)
       │       ↓
       │   [Otsu masking]  filtered_frame → 0–255 → cv2.threshold → dilate
       │       ↓  per-row search window [start : end]
       │   [Edge function]  filtered_frame[row, start:end] → edge_x_local → +start
       │       ↓
       │   frame_result[row] = edge_x_absolute
       │
       └─ edge_results  (T × H)   →   saved to HDF5 as edge_results/data
```

**What to try next:**
- Change `THRESHOLD_IR` and re-run cells 5–7 to see how sensitive the result is
- Change `FLAME_DIR` to `'left_to_right'` and compare L→R methods
- Swap `USE_SYNTHETIC = False` and load your own experiment
- Replace `custom_filter = lambda x: x` in cell 6 with `band_filter(x, low=250, high=600)` to see the effect of intensity clipping